In [ ]:
# this kernel combines the code in the other two but it might be less efficient because
# of the added function calls -- to be tested
potential = cp.RawKernel(r'''
extern "C" __global__
void potential(const double m[], const double q[][6], double q_dot[][6]) {
    double nucl_m = 1.71e9;
    double nucl_a = 0.07e3;
    double bulge_m = 5e9;
    double bulge_a = 1e3;
    double disk_m = 6.8e10;
    double disk_a = 3e3;
    double disk_b = 0.28e3;
    double halo_m = 5.4e11;
    double halo_a = 15.62e3;
    double G = %s;
    
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    int n = blockDim.y * blockIdx.y + threadIdx.y;
    
    if (i >= %d or n >= %d)
        return;
    
    if (n == 0) {
        double r = sqrt(q[i][0]*q[i][0] + q[i][1]*q[i][1] + q[i][2]*q[i][2]);
        double sqrtz = sqrt(q[i][2]*q[i][2] + disk_b*disk_b);
        double zd = disk_a + sqrtz;
        double tmp = G * disk_m * pow(q[i][0]*q[i][0] + q[i][1]*q[i][1] + zd*zd, -1.5);

        for (int j = 0; j < 3; j++) {
            q_dot[i][j] = q[i][j+3];

            q_dot[i][j+3] = -G * nucl_m / (r * (r+nucl_a) * (r+nucl_a)) * q[i][j];
            q_dot[i][j+3] -= G * bulge_m / (r * (r+bulge_a) * (r+bulge_a)) * q[i][j];
            q_dot[i][j+3] -= G * halo_m * (log(1 + r/halo_a) - r/(r + halo_a)) / (r*r*r) * q[i][j];

            q_dot[i][j+3] -= tmp * q[i][j] * ( j==2 ? (1 + disk_a/sqrtz) : 1);
        }
    }

    if (n == i)
        return;

    double r2 = 0;
    for (int k = 0; k < 3; k++)
        r2 += (q[i][k] - q[n][k]) * (q[i][k] - q[n][k]);

    for (int k = 0; k < 3; k++)
        q_dot[i][k+3] -= G * m[n] * (q[i][k] - q[n][k]) / (r2 * sqrt(r2));
}
''' % (G_flt, N, N), 'potential')

time_step_cuda = cp.RawKernel(r'''
extern "C" __global__
void time_step_cuda(const double m[], const double q[][6], double p[][6]) {
    int N = %d;
    double dt = %s;
    
    double q_dot1[N][6];
    milky_way_potential(q, q_dot1);
    n_body_potential(m, q, q_dot1);
    
    double q_tmp[N][6];
    for (int j = 0; j < N; j++)
        for (int k = 0; k < 6; k++)
            q_tmp[j][k] = q[j][k] + dt * q_dot1[j][k];
            
    double q_dot2[N][6];
    milky_way_potential(q_tmp, q_dot2);
    n_body_potential(m, q_tmp, q_dot2);
    
    for (int j = 0; j < N; j++)
        for (int k = 0; k < 6; k++)
            p[j][k] = q[j][k] + dt/2 * (q_dot1[j][k] + q_dot2[j][k]);
}
''' % (N, dt), 'time_step_cuda')